In [ ]:

# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES
# TO THE CORRECT LOCATION (/kaggle/input) IN YOUR NOTEBOOK,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

import os
import sys
from tempfile import NamedTemporaryFile
from urllib.request import urlopen
from urllib.parse import unquote, urlparse
from urllib.error import HTTPError
from zipfile import ZipFile
import tarfile
import shutil

CHUNK_SIZE = 40960
DATA_SOURCE_MAPPING = 'data-science-london-scikit-learn:https%3A%2F%2Fstorage.googleapis.com%2Fkaggle-competitions-data%2Fkaggle-v2%2F3428%2F31210%2Fbundle%2Farchive.zip%3FX-Goog-Algorithm%3DGOOG4-RSA-SHA256%26X-Goog-Credential%3Dgcp-kaggle-com%2540kaggle-161607.iam.gserviceaccount.com%252F20240618%252Fauto%252Fstorage%252Fgoog4_request%26X-Goog-Date%3D20240618T224039Z%26X-Goog-Expires%3D259200%26X-Goog-SignedHeaders%3Dhost%26X-Goog-Signature%3Db73f3443fa133909001332f032be7866a9a61d26d907838a2715707a93ccb6fe8db79a43014ef8e3b917ff704314e5010c023a83a7845d9ab57814ad9c9c2668a833cd190b917fff77ca80077749eac8c8a6dd76110f6e539cbfe7e10ff82c55903e4a850c104f8919e5699fe69e3e5c326e17dc5a6518f1cac337992c17c6e718ab352a61f11cf8591a03fbd54277254d923570bd12607b4554d559ab601a667213bcb204ccd32a61cf55d4fd5f01e0e02ee8a622ca079aa1a07ee69dd989e265bb8a78cbcf6b2e9854ded21d5fa4f57e2f635190397e78004cdd72345738657f75166d147d37ccf7360dbc811ccb0226fcfc96183a11d8af2a81a5edc5636b'

KAGGLE_INPUT_PATH='/kaggle/input'
KAGGLE_WORKING_PATH='/kaggle/working'
KAGGLE_SYMLINK='kaggle'

!umount /kaggle/input/ 2> /dev/null
shutil.rmtree('/kaggle/input', ignore_errors=True)
os.makedirs(KAGGLE_INPUT_PATH, 0o777, exist_ok=True)
os.makedirs(KAGGLE_WORKING_PATH, 0o777, exist_ok=True)

try:
  os.symlink(KAGGLE_INPUT_PATH, os.path.join("..", 'input'), target_is_directory=True)
except FileExistsError:
  pass
try:
  os.symlink(KAGGLE_WORKING_PATH, os.path.join("..", 'working'), target_is_directory=True)
except FileExistsError:
  pass

for data_source_mapping in DATA_SOURCE_MAPPING.split(','):
    directory, download_url_encoded = data_source_mapping.split(':')
    download_url = unquote(download_url_encoded)
    filename = urlparse(download_url).path
    destination_path = os.path.join(KAGGLE_INPUT_PATH, directory)
    try:
        with urlopen(download_url) as fileres, NamedTemporaryFile() as tfile:
            total_length = fileres.headers['content-length']
            print(f'Downloading {directory}, {total_length} bytes compressed')
            dl = 0
            data = fileres.read(CHUNK_SIZE)
            while len(data) > 0:
                dl += len(data)
                tfile.write(data)
                done = int(50 * dl / int(total_length))
                sys.stdout.write(f"\r[{'=' * done}{' ' * (50-done)}] {dl} bytes downloaded")
                sys.stdout.flush()
                data = fileres.read(CHUNK_SIZE)
            if filename.endswith('.zip'):
              with ZipFile(tfile) as zfile:
                zfile.extractall(destination_path)
            else:
              with tarfile.open(tfile.name) as tarfile:
                tarfile.extractall(destination_path)
            print(f'\nDownloaded and uncompressed: {directory}')
    except HTTPError as e:
        print(f'Failed to load (likely expired) {download_url} to path {destination_path}')
        continue
    except OSError as e:
        print(f'Failed to load {download_url} to path {destination_path}')
        continue

print('Data source import complete.')


# Блокнот с моделями: KNN, SVC, GradientBoostingClassifier

## Задание по предмету Принятие Управленческих решений на основе данных

*Введение в базовый ML: ПЗ**

1. Зарегистрируйтесь на платформе kaggle.com

2. Выберите одно из соревнований (можно как прошедших, так и идущих сейчас)

3. Сделайте 3 ноутбука, проведите EDA-анализ, на каждый ноутбук - разная модель, которые обсуждались на вебинаре, и не только (все релевантные из scikit-learn, так и совместимых c этим фреймворком). Чем нестандартнее вы выберете модель - тем лучше. В процессе выполнения использовать механизм пайплайнов.

4. Для каждого ноутбука сделайте сабмит в лидербоард.

5. Критерии - чистый код, EDA (оформление, графики, анализ, инсайты), метрика соревнования.

6. В качестве ответа - ссылка на ноутбук с сабмитом.

## Задание Data Sciece London Scikit learn
https://www.kaggle.com/competitions/data-science-london-scikit-learn/

План:
1. Провести EDA с необходимой визуализацией
2. Приметить модели машинного обучения
В данном ноутбуке будет применена модель KNN модель


In [ ]:
pip install catboost

In [ ]:
# Импортируем библиотеки
import numpy as np
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from catboost import CatBoostClassifier
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix


### Загружаем данные:

In [ ]:
train = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/train.csv',header=None)
test = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/test.csv',header=None)
train_labels = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/trainLabels.csv',header=None)

### Анализ данных:

In [ ]:
train.info()

In [ ]:
train.shape

In [ ]:
train.isna().sum().sum()

In [ ]:
test.isna().sum().sum()

In [ ]:
train.head()

### Визуализация:

In [ ]:
plt.figure(figsize=(10, 6))
for col in train.columns:
    plt.plot(train.index, train[col], label=f'Column {col}')
plt.title('Разброс данных')
plt.xlabel('Индекс')
plt.ylabel('Значение')
plt.show()

In [ ]:
# Разделим датасет на 2 части
df_part1 = train.iloc[:, :20]
df_part2 = train.iloc[:, 20:]

# График 1
plt.figure(figsize=(20, 12))
corr_part1 = df_part1.corr()
sns.heatmap(corr_part1, annot=True, cmap='cubehelix', vmin=-1, vmax=1, center=0)
plt.title('Корреляционная тепловая карта по DF1')
plt.show()

# График 2
plt.figure(figsize=(20, 12))
corr_part2 = df_part2.corr()
sns.heatmap(corr_part2, annot=True, cmap='cubehelix', vmin=-1, vmax=1, center=0)
plt.title('Корреляционная тепловая карта по DF2')
plt.show()

In [ ]:
# Бесконечные значения заменим на NaN
train.replace([np.inf, -np.inf], np.nan, inplace=True)

# Построим боксплоты
plt.figure(figsize=(10, 6))
sns.boxplot(data=train)
plt.title('Размах данных')
plt.show()

In [ ]:
x,y = train,train_labels
X_train,X_test,y_train,y_test = train_test_split(x,y,test_size=0.3,random_state=42)

### Применение моделей:

### Модель K-ближайших соседей (KNN):


In [ ]:
# Импортируем необходимые библиотеки для создания пайплайнов
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Создаем и обучаем пайплайн для модели K-ближайших соседей
pipeline_knn = Pipeline([
    ('scaler', StandardScaler()),  # Шаг масштабирования признаков
    ('classifier', KNeighborsClassifier())  # Шаг классификации
])

pipeline_knn.fit(X_train, y_train)  # Обучаем пайплайн

In [ ]:
# Используем пайплайн для предсказания и оценки точности
knn_pred = pipeline_knn.predict(X_test)
knn_accuracy = accuracy_score(y_test, knn_pred)

print(f'Точность модели K-ближайших соседей: {knn_accuracy * 100:.2f}%')



```
# Выбран кодовый формат
```

### Модель опорных векторов (SVC):


In [ ]:
# Импортируем необходимые библиотеки для создания пайплайнов
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Создаем и обучаем пайплайн для модели опорных векторов
pipeline_svc = Pipeline([
    ('scaler', StandardScaler()),  # Шаг масштабирования признаков
    ('classifier', SVC())  # Шаг классификации
])

pipeline_svc.fit(X_train, y_train)  # Обучаем пайплайн

In [ ]:
# Используем пайплайн для предсказания и оценки точности
svc_pred = pipeline_svc.predict(X_test)
svc_accuracy = accuracy_score(y_test, svc_pred)

print(f'Точность модели опорных векторов: {svc_accuracy * 100:.2f}%')

### Модель градиентного бустинга (GradientBoostingClassifier)

In [ ]:
# Импортируем необходимые библиотеки для создания пайплайнов
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

# Создаем и обучаем пайплайн для модели градиентного бустинга
pipeline_gb = Pipeline([
    ('scaler', StandardScaler()),  # Шаг масштабирования признаков
    ('classifier', GradientBoostingClassifier())  # Шаг классификации
])

pipeline_gb.fit(X_train, y_train)  # Обучаем пайплайн



In [ ]:
# Используем пайплайн для предсказания и оценки точности
gb_pred = pipeline_gb.predict(X_test)
gb_accuracy = accuracy_score(y_test, gb_pred)

print(f'Точность модели градиентного бустинга: {gb_accuracy * 100:.2f}%')